# 04 — Demo interactiva para póster

Explorador con **ipywidgets** para comparar predicciones de Ridge, Decision Tree y Random Forest.

**Requisito:** ejecutar antes el notebook **02** (genera `reports/metrics/model_metrics.csv`).

**Dataset:** [UCI Appliances Energy Prediction](https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction)

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

TARGET = "Appliances"
DATE_COL = "date"
DROP_COLS = ["rv1", "rv2"]
TEST_RATIO = 0.2
RANDOM_STATE = 42
USE_GRADIENT_BOOSTING = True

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "energydata_complete.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
METRICS_DIR = ROOT / "reports" / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Coloca energydata_complete.csv en {DATA_PATH}")

In [2]:
def load_and_clean(csv_path: Path) -> pd.DataFrame:
    """Carga el CSV UCI y aplica limpieza básica."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object").columns:
        if col != DATE_COL:
            df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    df = df.drop_duplicates()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Variables temporales, climáticas y de rezago (solo información pasada)."""
    out = df.copy()
    dt = out[DATE_COL]

    out["hour"] = dt.dt.hour
    out["day_of_week"] = dt.dt.dayofweek
    out["month"] = dt.dt.month
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["day_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["day_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)

    t_cols = [f"T{i}" for i in range(1, 10) if f"T{i}" in out.columns]
    rh_cols = [f"RH_{i}" for i in range(1, 10) if f"RH_{i}" in out.columns]
    out["T_inside_mean"] = out[t_cols].mean(axis=1)
    out["RH_inside_mean"] = out[rh_cols].mean(axis=1)
    out["delta_T_out_inside"] = out["T_out"] - out["T_inside_mean"]

    out["Appliances_lag_1"] = out[TARGET].shift(1)
    out["Appliances_lag_3"] = out[TARGET].shift(3)
    out["Appliances_lag_6"] = out[TARGET].shift(6)
    past = out[TARGET].shift(1)
    out["Appliances_roll_3"] = past.rolling(3, min_periods=3).mean()
    out["Appliances_roll_6"] = past.rolling(6, min_periods=6).mean()
    out["Appliances_roll_12"] = past.rolling(12, min_periods=12).mean()
    return out


def prepare_dataset(csv_path: Path) -> pd.DataFrame:
    """Limpieza, features y drop de NaN por lag/rolling."""
    df = add_features(load_and_clean(csv_path))
    before = len(df)
    df = df.dropna().reset_index(drop=True)
    print(f"Filas tras lag/rolling: {len(df):,} (eliminadas: {before - len(df):,})")
    return df


def temporal_train_test_split(df: pd.DataFrame, test_ratio: float = TEST_RATIO):
    """Partición cronológica 80/20."""
    split_idx = int(len(df) * (1 - test_ratio))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def get_xy(df: pd.DataFrame):
    y = df[TARGET]
    X = df.drop(columns=[TARGET, DATE_COL], errors="ignore")
    return X, y


def save_fig(name: str) -> None:
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figura guardada: {path}")

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor


def regression_metrics(y_true, y_pred) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}


def make_ridge_pipeline(feature_names):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), feature_names),
        ])),
        ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])


def make_tree_pipeline(estimator, feature_names):
    """Árboles: sin StandardScaler (no es necesario)."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), feature_names),
        ])),
        ("model", estimator),
    ])

In [4]:
metrics_df = pd.read_csv(METRICS_DIR / 'model_metrics.csv')
best_model = metrics_df.sort_values('rmse').iloc[0]['model']
print(f'Mejor modelo en test: {best_model}')

df = prepare_dataset(DATA_PATH)
train_df, test_df = temporal_train_test_split(df)
X_train, y_train = get_xy(train_df)
n_feat = len(X_train.columns)
print(f'Train: {len(X_train):,} registros | {n_feat} features')

Mejor modelo en test: Ridge
Filas tras lag/rolling: 19,723 (eliminadas: 12)
Train: 15,778 registros | 42 features


C:\Users\yulcr\AppData\Local\Temp\ipykernel_24292\4246567763.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


## Demo interactiva

Esta demo **no reemplaza el modelo completo**, sino que sirve como visualización interactiva para explicar cómo algunas variables afectan la predicción.

Ajusta los controles y compara en **tiempo real** las estimaciones de **Ridge**, **Decision Tree** y **Random Forest**.

In [5]:
from IPython.display import Markdown, display
import ipywidgets as widgets
from ipywidgets import Layout, Output, VBox

# Entrenar los 3 modelos para comparación en tiempo real
feature_names = list(X_train.columns)
model_factories = {
    "Ridge": lambda: make_ridge_pipeline(feature_names),
    "Decision Tree": lambda: make_tree_pipeline(
        DecisionTreeRegressor(
            max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE
        ),
        feature_names,
    ),
    "Random Forest": lambda: make_tree_pipeline(
        RandomForestRegressor(
            n_estimators=100,
            max_depth=16,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        feature_names,
    ),
}

fitted = {}
for name, factory in model_factories.items():
    pipe = factory()
    pipe.fit(X_train, y_train)
    fitted[name] = pipe

feature_template = X_train.median(numeric_only=True).to_frame().T
default_month = int(X_train["month"].median())

SLIDER_STYLE = {"description_width": "210px"}
SLIDER_LAYOUT = Layout(width="95%")
CONTINUOUS = {"continuous_update": True}


def classify_consumption(value: float) -> str:
    if value < 80:
        return "Bajo"
    if value <= 200:
        return "Medio"
    return "Alto"


def build_feature_row(
    hour,
    day_of_week,
    t_out,
    rh_out,
    t_inside_mean,
    rh_inside_mean,
    lights,
    appliances_lag_1,
    appliances_roll_3,
    appliances_roll_6,
) -> pd.DataFrame:
    sample = feature_template.copy()
    updates = {
        "hour": hour,
        "day_of_week": day_of_week,
        "month": default_month,
        "is_weekend": int(day_of_week >= 5),
        "T_out": t_out,
        "RH_out": rh_out,
        "T_inside_mean": t_inside_mean,
        "RH_inside_mean": rh_inside_mean,
        "delta_T_out_inside": t_out - t_inside_mean,
        "lights": lights,
        "Appliances_lag_1": appliances_lag_1,
        "Appliances_roll_3": appliances_roll_3,
        "Appliances_roll_6": appliances_roll_6,
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "day_sin": np.sin(2 * np.pi * day_of_week / 7),
        "day_cos": np.cos(2 * np.pi * day_of_week / 7),
    }
    for column, value in updates.items():
        if column in sample.columns:
            sample[column] = value
    return sample


def predict_all_models(sample: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, pipe in fitted.items():
        pred = float(pipe.predict(sample)[0])
        rows.append(
            {
                "Modelo": name,
                "Consumo (Wh)": round(pred, 2),
                "Nivel": classify_consumption(pred),
                "Es mejor en test": name == best_model,
            }
        )
    return pd.DataFrame(rows).sort_values("Consumo (Wh)").reset_index(drop=True)


# Sliders clásicos (barra deslizante) — todos visibles en la misma vista
w_hour = widgets.IntSlider(
    value=18, min=0, max=23, step=1, description="Hora del día (0-23 h)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_day = widgets.Dropdown(
    options=[
        ("Lunes", 0), ("Martes", 1), ("Miércoles", 2), ("Jueves", 3),
        ("Viernes", 4), ("Sábado", 5), ("Domingo", 6),
    ],
    value=0, description="Día de la semana",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT,
)
w_t_out = widgets.FloatSlider(
    value=8.0, min=-5.0, max=30.0, step=0.5, description="Temp. exterior (°C)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_rh_out = widgets.FloatSlider(
    value=80.0, min=20.0, max=100.0, step=1.0, description="Humedad exterior (%)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_t_in = widgets.FloatSlider(
    value=21.0, min=15.0, max=30.0, step=0.5, description="Temp. interior media (°C)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_rh_in = widgets.FloatSlider(
    value=40.0, min=20.0, max=70.0, step=1.0, description="Humedad interior media (%)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_lights = widgets.IntSlider(
    value=0, min=0, max=70, step=10, description="Uso de luces",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_lag1 = widgets.IntSlider(
    value=60, min=10, max=600, step=10, description="Consumo hace 10 min (Wh)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_roll3 = widgets.IntSlider(
    value=70, min=10, max=600, step=10, description="Media últimos 30 min (Wh)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)
w_roll6 = widgets.IntSlider(
    value=80, min=10, max=600, step=10, description="Media últimos 60 min (Wh)",
    style=SLIDER_STYLE, layout=SLIDER_LAYOUT, **CONTINUOUS,
)

all_controls = [
    w_hour, w_day, w_t_out, w_rh_out, w_t_in, w_rh_in,
    w_lights, w_lag1, w_roll3, w_roll6,
]

demo_output = Output()


def refresh_demo(_=None):
    sample = build_feature_row(
        w_hour.value, w_day.value, w_t_out.value, w_rh_out.value,
        w_t_in.value, w_rh_in.value, w_lights.value,
        w_lag1.value, w_roll3.value, w_roll6.value,
    )
    comparison = predict_all_models(sample)
    ref_pred = float(
        comparison.loc[comparison["Modelo"] == best_model, "Consumo (Wh)"].iloc[0]
    )
    ref_level = classify_consumption(ref_pred)
    spread = comparison["Consumo (Wh)"].max() - comparison["Consumo (Wh)"].min()

    demo_output.clear_output(wait=True)
    with demo_output:
        display(Markdown(
            f"### Comparación entre modelos\n"
            f"Mejor modelo en test (**{best_model}**): **{ref_pred:.2f} Wh** ({ref_level}). "
            f"Diferencia máxima entre modelos: **{spread:.2f} Wh**."
        ))
        display(
            comparison.style.hide(axis="index").background_gradient(
                subset=["Consumo (Wh)"], cmap="Blues"
            )
        )

        fig, ax = plt.subplots(figsize=(8, 3))
        colors = ["C2" if m == best_model else "C0" for m in comparison["Modelo"]]
        bars = ax.barh(comparison["Modelo"], comparison["Consumo (Wh)"], color=colors)
        ax.axvline(80, linestyle="--", linewidth=1, alpha=0.8)
        ax.axvline(200, linestyle="--", linewidth=1, alpha=0.8)
        ax.set_xlabel("Consumo estimado (Wh)")
        ax.set_title("Estimación por modelo (verde = mejor en test)")
        for bar, nivel in zip(bars, comparison["Nivel"]):
            ax.text(
                bar.get_width() + 1,
                bar.get_y() + bar.get_height() / 2,
                nivel,
                va="center",
                fontsize=9,
            )
        plt.tight_layout()
        plt.show()

        display(Markdown(
            f"**Lectura rápida:** consumo **{ref_level.lower()}** según {best_model}. "
            f"Suele subir si aumentas **consumo hace 10 min** o **luces**."
        ))


for widget in all_controls:
    widget.observe(refresh_demo, names="value")

guide_md = widgets.HTML(
    "<p style='margin:0 0 10px 0; color:#444'>"
    "<b>Controles:</b> hora y día = patrón temporal; temp./humedad interior y exterior = clima; "
    "luces y consumo reciente (10 min, media 30 y 60 min) = uso del hogar. "
    "Otras columnas del modelo = mediana del entrenamiento.</p>"
)

controls_box = VBox(
    [guide_md] + all_controls,
    layout=Layout(width="100%"),
)

demo_ui = VBox([controls_box, demo_output])
display(demo_ui)
refresh_demo()